# Learned marine space–time transport

This research-only experiment asks whether ecotype evidence should be transported between sightings using learned water-distance, elapsed-time, implied-speed, and temporal-direction weights instead of one fixed kernel. It reconstructs the current encounter-held-out folds and removes every outer-test encounter from the anchor pool before feature generation.

The result is an evidence-transport model, not an animal trajectory model. No inferred route becomes an observation or count. All outputs stay under `outputs/transport/`, and no candidate is production-promoted here.

In [ ]:
import os
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from experiment_support import resolve_release_paths
from transport_experiment import run_transport_experiment, transport_kernel_grid

paths = resolve_release_paths()
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'transport'
paths

## Execute the complete experiment

Thirty-two kernels span four spatial scales, four temporal scales, and optional speed decay. Support is calculated independently for SRKW, Transient, and Other anchors and for past, same-day, and future directions. Anchor reports are collapsed to encounter-level maxima before summation.

In [ ]:
experiment = run_transport_experiment(output_dir, paths)
manifest = json.loads(experiment["manifest_path"].read_text())
manifest["recommended_research_challenger"]

## Aggregate probability performance

Metrics use SOURCE × observed-class encounter-prevalence weights. Lower Brier, log loss, ECE, and calibration-intercept magnitude are better; calibration slope should be close to one. The past-only variant is the deployable-direction analogue, whereas the retrospective variant may use future observations.

In [ ]:
comparison = experiment["transport_comparison"].set_index("model")
display(comparison.round(5))
bootstrap = pd.DataFrame(experiment["bootstrap"]).T
display(bootstrap.round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison[["brier", "log_loss"]].plot.bar(ax=axes[0], rot=35, title="Probability loss")
comparison[["equal_mass_ece_10"]].plot.bar(ax=axes[1], rot=35, legend=False, title="Equal-mass calibration error")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## Source-level safety gates

Aggregate improvement is insufficient. A candidate fails when Brier score degrades by more than 10% for any source with at least 100 independent evaluation encounters. Source is used only for weighting, diagnostics, and this promotion guard—not as a prediction feature.

In [ ]:
display(experiment["transport_source_gate_summary"].sort_values(["all_required_sources_pass", "maximum_relative_brier_degradation"], ascending=[False, True]).round(5))
recommended = manifest["recommended_research_challenger"]
gate_detail = experiment["transport_source_gates"]
display(gate_detail.loc[gate_detail.model.isin(["learned_multiscale_retrospective", recommended])].round(5))

## What the model learned

The sparse logistic layer combines monotonic kernel bases. Large coefficients identify influential scales and directions, but correlated bases mean individual coefficients should be read as diagnostic rather than causal movement estimates.

In [ ]:
coefficients = experiment["transport_top_coefficients"]
stable_features = (coefficients.groupby(["variant", "feature"], as_index=False).agg(mean_absolute_coefficient=("absolute_coefficient", "mean"), folds_selected=("outer_fold", "nunique")).sort_values(["variant", "folds_selected", "mean_absolute_coefficient"], ascending=[True, False, False]))
display(stable_features.groupby("variant").head(15).round(5))
display(experiment["transport_fold_selections"].round(5))
display(experiment["transport_support_diagnostics"].round(5))

## Decision boundary

The recommended research challenger is selected only among models that pass every required source gate. It still cannot be promoted: exact incumbent comparison currently has three outer folds, the sample is capped, the biological target remains closed-set, rolling-origin/spatial/source holdouts are incomplete, and no blinded unknown-label audit exists. The next production-oriented step is to integrate the constrained blend into five-fold rolling and repeated spatial evaluation—not to enable soft counts.